# Artisan Hub Flask Backend Integration

In [ ]:
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)

artisan_pkg = joblib.load('artisan_lead_payment_model.pkl')
customer_pkg = joblib.load('customer_premium_model.pkl')

def encode_input(df, package):
    for col, le in package['label_encoders'].items():
        if col in df.columns:
            df[col]=df[col].apply(lambda x: x if x in le.classes_ else le.classes_[0])
            df[col]=le.transform(df[col].astype(str))
    for feat in package['features']:
        if feat not in df.columns:
            df[feat]=0
    X=df[package['features']].values
    return package['scaler'].transform(X)

@app.route('/predict/artisan', methods=['POST'])
def predict_artisan():
    data=request.get_json()
    df=pd.DataFrame([data])
    for col in df.columns:
        df[col]=df[col].apply(lambda x:str(x).strip().lower() if pd.notna(x) else 'unknown')
    X=encode_input(df,artisan_pkg)
    prob=artisan_pkg['model'].predict_proba(X)[0,1]
    pred=int(artisan_pkg['model'].predict(X)[0])
    return jsonify({
        'will_pay_for_leads': bool(pred),
        'confidence': round(float(prob),3),
        'segment':'HIGH' if prob>0.7 else 'MEDIUM' if prob>0.4 else 'LOW',
        'recommended_action':'Fast-track onboarding' if prob>0.7 else 'Trial period' if prob>0.4 else 'Free tier first'
    })

@app.route('/predict/customer', methods=['POST'])
def predict_customer():
    data=request.get_json()
    df=pd.DataFrame([data])
    for col in df.columns:
        df[col]=df[col].apply(lambda x:str(x).strip().lower() if pd.notna(x) else 'unknown')
    X=encode_input(df,customer_pkg)
    prob=customer_pkg['model'].predict_proba(X)[0,1]
    pred=int(customer_pkg['model'].predict(X)[0])
    return jsonify({
        'will_pay_premium': bool(pred),
        'confidence': round(float(prob),3),
        'ltv_segment':'HIGH' if prob>0.7 else 'MEDIUM' if prob>0.4 else 'LOW',
        'pricing_tier':'Premium' if prob>0.7 else 'Standard' if prob>0.4 else 'Freemium'
    })

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status':'ok','models_loaded':True})

if __name__=='__main__':
    app.run(debug=True, use_reloader=False, port=5000)


## Test API

Run:

```bash
python app.py
```

Health:
`GET http://127.0.0.1:5000/health`

Artisan:
`POST http://127.0.0.1:5000/predict/artisan`

Customer:
`POST http://127.0.0.1:5000/predict/customer`
